# celldform — end-to-end pipeline

Cancer-cell deformation under optical tweezers. This notebook walks the whole pipeline,
**acquisition → preprocessing → segmentation → features → classification → biomechanics
→ real-time inference**, against the data actually on disk.

## How to use it

Run top to bottom for a guided tour: most cells only **read and display**. The handful
that write or overwrite data are each gated behind an explicit flag:

```python
RUN_THIS = False   # ← flip to True to actually execute
```

Those cells print what they *would* do while the flag is `False`. **Leave them off
unless you mean it** — a couple of them rewrite the frame pools, which is how the
17 September cell exchange had to be undone once already.

## Three project rules the notebook assumes

1. **A single-run difference below ~0.13 DSC is not a result.** Measured seed-only
   spread is sd 0.047. See the evaluation section.
2. **Change one variable per run.** If a non-experimental setting moves (schedule,
   split, width, pool), re-baseline before quoting a delta.
3. **Check the split log line** at the top of every training run — `high_cell10` is a
   50-frame group against a ~45-frame validation target and lands in validation on 13%
   of seeds. When it does, that run is void.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, sys, json, subprocess, warnings
from pathlib import Path
from collections import Counter, defaultdict

REPO = Path.cwd()
if REPO.name == "notebooks":            # allow running from notebooks/
    REPO = REPO.parent
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "image.cmap": "gray", "figure.facecolor": "white"})

PY = str(REPO / ".venv" / "bin" / "python")
if not Path(PY).exists():
    PY = sys.executable                 # fall back to whatever kernel is running

def sh(*args, show=True):
    """Run a repo script and stream its output. Returns (returncode, stdout)."""
    r = subprocess.run([PY, *map(str, args)], cwd=REPO,
                       capture_output=True, text=True)
    out = r.stdout + ("\n" + r.stderr if r.returncode else "")
    if show:
        print(out[-4000:] if len(out) > 4000 else out)
    return r.returncode, out

print(f"repo      : {REPO}")
print(f"python    : {PY}")
print(f"torch     : {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu       : {p.name}  {p.total_memory/1e9:.1f} GB")

---
## 1 · Where everything lives

The repo is split `preswap/` vs `postswap/` — the **17 September 2026 cell exchange**
moved `high-10` into training and `high-1` into evaluation. Numbers either side of it
are **not comparable**, so the folders keep them apart.

In [ ]:
# ── Layout and the current baseline ───────────────────────────────────────────
for area in ("configs", "checkpoints", "logs", "results"):
    subs = sorted(p.name for p in (REPO/area).iterdir() if p.is_dir())
    print(f"{area+'/':<14}{'  '.join(subs)}")
print(f"{'scripts/':<14}" + "  ".join(sorted(p.name for p in (REPO/'scripts').iterdir() if p.is_dir())))

BASELINE_CFG  = REPO/"configs/postswap/postswap_ftl_cw142_s42.yaml"
BASELINE_CKPT = REPO/"checkpoints/postswap/postswap_ftl_cw142_s42/unet_best.pt"
POST_LOG      = REPO/"results/postswap/postswap_eval_log.jsonl"
PRE_LOG       = REPO/"results/preswap/preswap_eval_log.jsonl"

print(f"\n{'BASELINE CONFIG':<18}{BASELINE_CFG.relative_to(REPO)}   exists={BASELINE_CFG.exists()}")
print(f"{'BASELINE CKPT':<18}{BASELINE_CKPT.relative_to(REPO)}   exists={BASELINE_CKPT.exists()}")
print("\nSee configs/README.md and checkpoints/README.md for what every other file is.")

In [ ]:
# ── The config that everything branches from ──────────────────────────────────
from celldform.config import load as load_config
conf = load_config(BASELINE_CFG)

print("data")
print(f"   frames_dir   {conf.data.frames_dir}")
print(f"   masks_dir    {conf.data.masks_dir}")
print(f"   val_split    {conf.data.val_split}")
print("unet")
print(f"   base_features {conf.unet.base_features}   out_channels {conf.unet.out_channels}   dropout {conf.unet.dropout}")
print("training")
for k in ("epochs","batch_size","lr","optimizer","scheduler","loss_type","tversky_alpha",
          "tversky_beta","focal_gamma","class_weights","seed","split_seed",
          "augment_scale_prob","augment_noise_std","early_stopping_patience"):
    print(f"   {k:<24}{getattr(conf.training, k)}")

---
## 2 · Inspect the data

Four pools. The rule that never bends: **a cell held out for evaluation appears
nowhere else.** The check below is the one to re-run after any pool change.

In [ ]:
# ── Pool composition ──────────────────────────────────────────────────────────
import re
FR, MK = REPO/"data/frames", REPO/"data/masks"
POOLS = {"01_annotate_pool": "U-Net training",
         "02_unet_holdout":  "development set (evaluation)",
         "03_mask_factory":  "classifier feature extraction"}

def cells_in(d, pattern="*.jpg"):
    c = Counter()
    for p in d.glob(pattern):
        s = p.stem.replace("domain_", "")
        m = re.match(r"((?:high|low)_cell\d+)", s)
        c[m.group(1) if m else "legacy"] += 1
    return c

rows = []
for pool, role in POOLS.items():
    d = FR/pool
    if not d.exists(): continue
    c = cells_in(d)
    rows.append({"pool": pool, "role": role, "frames": sum(c.values()), "groups": len(c)})
    print(f"\n{pool}  —  {role}")
    print("   " + "  ".join(f"{k}:{v}" for k, v in sorted(c.items())))
display(pd.DataFrame(rows))

In [ ]:
# ── FIREWALL CHECK — run this after ANY pool change ───────────────────────────
DEV_CELLS = ["high_cell1", "high_cell9", "low_cell5", "low_cell6"]
violations = []
for cell in DEV_CELLS:
    n = len(list((FR/"01_annotate_pool").glob(f"*{cell}_*.jpg"))) \
      + len(list((FR/"03_mask_factory").glob(f"{cell}_*.jpg")))
    status = "OK" if n == 0 else "*** VIOLATION ***"
    if n: violations.append(cell)
    print(f"  {cell:<12}{n:>3} frames in training pools   {status}")

# masks must match frames, and every manifest row must point inside its own pool
import csv
print()
for pool, mdir in (("01_annotate_pool","01_annotate_pool_multiclass"),
                   ("02_unet_holdout","02_unet_holdout_multiclass")):
    f = len(list((FR/pool).glob("*.jpg"))); m = len(list((MK/mdir).glob("*.png")))
    print(f"  {pool:<18}{f:>4} frames / {m:>4} masks" + ("" if f == m else f"   ({f-m} unannotated)"))

bad = 0
for man in FR.rglob("00_*manifest*.csv"):
    for r in csv.DictReader(man.open()):
        if "output_path" in r and Path(r["output_path"]).parent != man.parent:
            bad += 1
print(f"  manifest rows pointing outside their pool: {bad}")
assert not violations, f"FIREWALL BROKEN: {violations}"
print("\n  firewall intact")

In [ ]:
# ── Look at some frames, with their masks ─────────────────────────────────────
def show_with_mask(pool, mask_dir, n=4, cell=None, title=""):
    frames = sorted((FR/pool).glob(f"*{cell}_*.jpg" if cell else "*.jpg"))
    if not frames:
        print(f"no frames for {cell} in {pool}"); return
    idx = np.linspace(0, len(frames)-1, min(n, len(frames))).astype(int)
    fig, ax = plt.subplots(2, len(idx), figsize=(3.1*len(idx), 6.2))
    ax = np.atleast_2d(ax)
    for j, i in enumerate(idx):
        fp = frames[i]
        img = cv2.imread(str(fp), cv2.IMREAD_GRAYSCALE)
        ax[0, j].imshow(img); ax[0, j].set_title(fp.stem[-13:], fontsize=8)
        mp = MK/mask_dir/f"{fp.stem}.png"
        if mp.exists():
            m = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
            rgb = np.dstack([img]*3).astype(float)/255
            rgb[m == 1] = [0.0, 0.9, 0.0]      # trapped cell
            rgb[m == 2] = [0.9, 0.0, 0.0]      # decoy / untrapped
            ax[1, j].imshow(rgb)
            ax[1, j].set_title(f"cell {int((m==1).sum())} px · decoy {int((m==2).sum())} px", fontsize=7)
        else:
            ax[1, j].text(.5, .5, "no mask", ha="center"); ax[1, j].set_facecolor("0.9")
    for a in ax.ravel(): a.axis("off")
    fig.suptitle(title or f"{pool} — {cell or 'all'}   (green = trapped cell, red = decoy)", fontsize=10)
    plt.tight_layout(); plt.show()

show_with_mask("02_unet_holdout", "02_unet_holdout_multiclass", cell="high_cell9",
               title="high-9 — the hard high-power cell (decoy confusion)")

In [ ]:
# ── Object scale: the finding that drives current direction ───────────────────
# After the 17x downsample to 256x256 the trapped cell is a ~6x6 to 9x9 px blob.
raw_px = 1228*922
rows = []
for cell in DEV_CELLS:
    cp, dp = [], []
    for p in (MK/"02_unet_holdout_multiclass").glob(f"{cell}_*.png"):
        m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        cp.append(int((m==1).sum()))
        d = int((m==2).sum())
        if d: dp.append(d)
    if not cp: continue
    mc = np.mean(cp)
    rows.append({"cell": cell.replace("_cell","-"), "raw px": round(mc),
                 "@256": round(mc*65536/raw_px), "side @256": f"~{np.sqrt(mc*65536/raw_px):.0f}px",
                 "decoy @256": round(np.mean(dp)*65536/raw_px) if dp else 0})
display(pd.DataFrame(rows))
print("Resolution is the untried lever best matched to this — see docs/status.md.")

---
## 3 · Stage 0 — organise raw video into pools  ⚠️ WRITES

`organise()` rebuilds `data/frames/` from `data/raw/`. **It is destructive** — with
`clean=True` it wipes the frame pools first, which would discard the 17 September
roster and every annotation alignment that depends on it.

Only run this when starting from raw video. Everything downstream already exists.

In [ ]:
# ── Stage 0 (GUARDED) ─────────────────────────────────────────────────────────
RUN_ORGANISE = False     # ← destructive: rebuilds data/frames/ from data/raw/

from celldform.acquisition.organiser import organise
import inspect
print(inspect.signature(organise), "\n")

if RUN_ORGANISE:
    organise(raw_dir=REPO/"data/raw", out_dir=REPO/"data/frames",
             every_n=200, domain_adapt_n=20, clean=False)   # clean=False: do not wipe
else:
    print("SKIPPED (RUN_ORGANISE = False).")
    print("Would rebuild the four pools under data/frames/ from data/raw/.")
    print("Existing pools:", ", ".join(p.name for p in sorted(FR.iterdir()) if p.is_dir()))
    print("\nNOTE: the current roster is the result of a deliberate 17 Sept exchange")
    print("recorded in data/SWAP_UNDO_20260917.json. Re-organising discards it.")

---
## 4 · Stage 1 — extract frames from one video

Safe: writes to a scratch directory, not the pools.

`change_threshold` must stay `None` for these experiments. The cell is *trapped* and
does not move; deformation is shape change, not displacement. A threshold makes the
static centroid filter out every frame.

In [ ]:
# ── Stage 1 — FrameExtractor on a single video ────────────────────────────────
from celldform.acquisition.extractor import FrameExtractor

videos = sorted((REPO/"data/raw").rglob("*.mp4"))
print(f"{len(videos)} raw videos. First few:")
for v in videos[:4]: print("   ", v.relative_to(REPO))

SCRATCH = REPO/"results/_notebook_scratch/frames"
if videos:
    fe = FrameExtractor(videos[0], output_dir=SCRATCH, target_size=(1228, 922),
                        image_format="jpg",
                        change_threshold=None)   # ← must be None: trapped cell does not move
    frames = fe.extract(every_n=2000)            # sparse: this is a demo, not the real grid
    print(f"\nextracted {len(frames)} frames -> {SCRATCH.relative_to(REPO)}")
    if frames:
        fig, ax = plt.subplots(1, len(frames), figsize=(3*len(frames), 3))
        for a, f in zip(np.atleast_1d(ax), frames):
            a.imshow(f); a.axis("off")
        plt.suptitle(f"{videos[0].name} — laser power rises left to right", fontsize=9)
        plt.tight_layout(); plt.show()

---
## 5 · Validate masks — the gate before preprocessing

Five checks per mask: present, readable, labels in {0,1,2}, non-empty, and **at least
one trapped-cell pixel**. That last one caught three genuinely cell-free frames
(frame 0, before the cell was trapped) that would otherwise have trained the model on
empty labels.

In [ ]:
# ── Validate both pools ───────────────────────────────────────────────────────
for pool in ("01_annotate_pool", "02_unet_holdout"):
    print(f"── {pool} " + "─"*46)
    sh("scripts/annotation/validate_masks.py", "--pool", pool, "--multiclass",
       *(["--no-completeness"] if pool == "02_unet_holdout" else []))

---
## 6 · Stage 2 — preprocessing

Preprocessing is **manual and separate** from training: `CellDataset` loads
already-preprocessed PNGs and applies nothing. Masks resize with `INTER_NEAREST` to
preserve the 0/1/2 label values — never linear or cubic.

In [ ]:
# ── What preprocessing does to one frame ──────────────────────────────────────
from celldform.preprocessing.pipeline import PreprocessingPipeline, PreprocessingConfig

pre = PreprocessingPipeline(PreprocessingConfig(
    median_kernel=conf.preprocessing.median_kernel,
    clahe_clip_limit=conf.preprocessing.clahe_clip_limit,
    clahe_tile_grid=tuple(conf.preprocessing.clahe_tile_grid),
    morph_kernel_size=conf.preprocessing.morph_kernel_size,
    output_size=tuple(conf.preprocessing.target_size),
))

src = sorted((FR/"02_unet_holdout").glob("high_cell9_*.jpg"))[10]
raw = cv2.imread(str(src), cv2.IMREAD_GRAYSCALE)
out = pre(raw)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(raw);  ax[0].set_title(f"raw  {raw.shape}")
ax[1].imshow(out);  ax[1].set_title(f"preprocessed  {out.shape}\n(median → CLAHE → morph → resize → normalise)")
ax[2].hist(raw.ravel(), bins=60, alpha=.6, label="raw", density=True)
ax[2].hist((out*255 if out.max() <= 1 else out).ravel(), bins=60, alpha=.6, label="after", density=True)
ax[2].legend(); ax[2].set_title("intensity distribution")
for a in ax[:2]: a.axis("off")
plt.tight_layout(); plt.show()
print(f"{raw.size/out.size:.1f}x fewer pixels — the trapped cell ends up ~6-9 px across.")

In [ ]:
# ── Batch preprocess both pools (GUARDED — overwrites preprocessed arrays) ────
RUN_PREPROCESS = False   # ← rewrites data/preprocessed/**; needed after ANY pool or mask change

cmds = [
    ["scripts/data/preprocess_frames.py", "--pool","01_annotate_pool","--stage","train","--masks",
     "--mask-dir","data/masks/01_annotate_pool_multiclass",
     "--output-dir","data/preprocessed/train/postswap_multiclass_256/inputs",
     "--mask-output-dir","data/preprocessed/train/postswap_multiclass_256/labels",
     "--target-size","256","256"],
    ["scripts/data/preprocess_frames.py", "--pool","02_unet_holdout","--stage","evaluate","--masks",
     "--mask-dir","data/masks/02_unet_holdout_multiclass",
     "--output-dir","data/preprocessed/evaluate/postswap_devset/inputs",
     "--mask-output-dir","data/preprocessed/evaluate/postswap_devset/labels"],
]
if RUN_PREPROCESS:
    # preprocess_frames.py does NOT clean stale outputs — wipe first or a removed cell
    # lingers and can end up in BOTH pools at once. This bit us on 17 September.
    import shutil
    for d in ("data/preprocessed/train/postswap_multiclass_256",
              "data/preprocessed/evaluate/postswap_devset"):
        shutil.rmtree(REPO/d, ignore_errors=True)
    for c in cmds: sh(*c)
else:
    print("SKIPPED (RUN_PREPROCESS = False). Current state:")
    for d in sorted((REPO/"data/preprocessed").rglob("inputs")):
        print(f"   {len(list(d.glob('*.png'))):>4} files   {d.parent.relative_to(REPO)}")
    print("\n⚠️  preprocess_frames.py appends; it does not clean. After a pool change,")
    print("    DELETE the output dirs first or stale frames survive in both pools.")

---
## 7 · Stage 3 — train the U-Net

Use `run_postswap.sh`: it trains, evaluates on all four development cells, files every
artefact in the right folder, and runs the baseline comparison. Calling `train_unet.py`
directly skips all of that.

**Before launching a new experiment:** copy the baseline config, change **one** field,
name it `postswap_<loss>_<cw>[_<newvar>]_s<seed>.yaml`, put it in `configs/postswap/`.

In [ ]:
# ── Inspect the training split WITHOUT training ───────────────────────────────
sys.path.insert(0, str(REPO/"scripts/training"))
from train_unet import _build_groups, _group_split

inp = Path(conf.data.frames_dir); lab = Path(conf.data.masks_dir)
pairs  = [(p, lab/p.with_suffix(".png").name) for p in sorted(inp.glob("*.png"))]
groups = _build_groups(pairs)
split_seed = conf.training.split_seed or conf.training.seed
tr, va = _group_split(pairs, groups, int(len(pairs)*conf.data.val_split), split_seed=split_seed)

vset = set(va)
vg = Counter(g for pr, g in zip(pairs, groups) if pr in vset)
tg = Counter(g for pr, g in zip(pairs, groups) if pr not in vset)
print(f"pool {len(pairs)} frames · {len(set(groups))} groups · split_seed {split_seed}")
print(f"train {len(tr)}   val {len(va)}\n")
print("val  :", ", ".join(f"{k}({v})" for k, v in vg.most_common()))
print("train:", ", ".join(f"{k}({v})" for k, v in tg.most_common() if "cell" in k))

big = [(k, v) for k, v in vg.items() if v >= 0.5*len(va)]
print("\n" + ("⚠️  " + str(big) + " is most of validation — this run would be VOID"
      if big else "healthy draw — no group dominates validation"))

### 7a · Build a config for this run

`configs/postswap/postswap_ftl_cw142_s42.yaml` is the **baseline** — every real
experiment branches from it and it should not be edited in place. Instead, load it,
override what you want, and write a **notebook-scoped** copy.

Everything a notebook run produces lands under `*/notebook/`, never in `preswap/` or
`postswap/`. That keeps exploratory runs out of the experimental record that
`compare_postswap.py` and the report tables are built from.

> **Sandbox results are not evidence.** If a notebook run produces something that
> matters, promote it: write a proper config into `configs/postswap/` and re-run it via
> `scripts/training/run_postswap.sh`, so it lands in the record with a clean provenance.


In [ ]:

# ── Notebook sandbox: load the baseline config, override, write a scoped copy ──
import yaml, datetime, copy

NB   = "notebook"                                   # sandbox namespace
NB_CFG_DIR  = REPO/"configs"/NB;      NB_CFG_DIR.mkdir(parents=True, exist_ok=True)
NB_CKPT_DIR = REPO/"checkpoints"/NB;  NB_CKPT_DIR.mkdir(parents=True, exist_ok=True)
NB_LOG_DIR  = REPO/"logs"/NB;         NB_LOG_DIR.mkdir(parents=True, exist_ok=True)
NB_RES_DIR  = REPO/"results"/NB;      NB_RES_DIR.mkdir(parents=True, exist_ok=True)
NB_EVAL_LOG = NB_RES_DIR/"notebook_eval_log.jsonl"

BASELINE_YAML = yaml.safe_load(BASELINE_CFG.read_text())

def make_config(run_name, **overrides):
    """Write configs/notebook/<run_name>.yaml = baseline + overrides.

    Override with dotted keys, e.g.:
        make_config("nb_lr3e4", training__lr=3e-4)
        make_config("nb_res384", preprocessing__target_size=[384, 384])
        make_config("nb_cw133",  training__class_weights=[1.0, 3.0, 3.0])

    Returns the path to the new config. checkpoint_dir is forced into
    checkpoints/notebook/ so a sandbox run can never overwrite a real checkpoint.
    """
    cfg = copy.deepcopy(BASELINE_YAML)
    applied = {}
    for key, val in overrides.items():
        parts = key.split("__")
        node = cfg
        for p in parts[:-1]:
            node = node.setdefault(p, {})
        before = node.get(parts[-1], "<unset>")
        node[parts[-1]] = val
        applied[".".join(parts)] = (before, val)

    cfg["checkpoint_dir"] = f"checkpoints/{NB}/{run_name}"      # forced — never a real dir

    header = (
        f"# GENERATED IN A NOTEBOOK — {datetime.datetime.now():%Y-%m-%d %H:%M}\n"
        f"# Derived from: {BASELINE_CFG.relative_to(REPO)}\n"
        f"# Overrides:\n" +
        ("".join(f"#     {k}: {b!r} -> {v!r}\n" for k, (b, v) in applied.items())
         if applied else "#     (none — an exact copy of the baseline)\n") +
        "#\n"
        "# Sandbox config. Results go to results/notebook/ and are NOT part of the\n"
        "# experimental record. To promote a finding, write a real config into\n"
        "# configs/postswap/ and run it with scripts/training/run_postswap.sh.\n"
        "# NOTE: the baseline's explanatory comments are lost in this round-trip —\n"
        "# read the original before changing something you do not recognise.\n\n"
    )
    out = NB_CFG_DIR/f"{run_name}.yaml"
    out.write_text(header + yaml.safe_dump(cfg, sort_keys=False, default_flow_style=False))

    print(f"wrote {out.relative_to(REPO)}")
    for k, (b, v) in applied.items():
        print(f"   {k:<34}{b!r}  ->  {v!r}")
    if not applied:
        print("   (no overrides — exact copy of the baseline)")
    return out

print("sandbox ready:")
for d in (NB_CFG_DIR, NB_CKPT_DIR, NB_LOG_DIR, NB_RES_DIR):
    print(f"   {d.relative_to(REPO)}")


In [ ]:

# ── Inspect and edit the config interactively ─────────────────────────────────
# Everything you can override, with its current value:
def show_config(d, prefix=""):
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            show_config(v, f"{key}__")
        else:
            print(f"   {key:<44}{v!r}")

show_config(BASELINE_YAML)
print("\nUse the '__' form as a keyword to make_config(), e.g. training__lr=3e-4")


In [ ]:

# ── Make a config for this run — EDIT THE OVERRIDES BELOW ─────────────────────
RUN_NAME = "nb_demo"

CFG_PATH = make_config(
    RUN_NAME,
    # --- examples; delete or change freely -----------------------------------
    training__epochs      = 6,          # short, so the sandbox run finishes quickly
    training__seed        = 42,
    training__split_seed  = 42,         # keep pinned unless you are probing the split
    # training__lr                 = 3e-4,
    # training__class_weights      = [1.0, 3.0, 3.0],
    # training__loss_type          = "dice_ce",
    # unet__base_features          = 32,
    # data__val_split              = 0.15,
)

# sanity: it loads through the real parser, so a typo fails here rather than mid-training
from celldform.config import load as _load
_cfg = _load(CFG_PATH)
print(f"\nparses OK · epochs={_cfg.training.epochs} lr={_cfg.training.lr} "
      f"loss={_cfg.training.loss_type} cw={_cfg.training.class_weights}")
print(f"checkpoints -> {_cfg.checkpoint_dir}")
assert f"/{NB}/" in str(_cfg.checkpoint_dir), "sandbox guard failed"


In [ ]:

# ── Train + evaluate inside the sandbox (GUARDED) ─────────────────────────────
RUN_TRAINING = False          # ← flip to True. With epochs=6 this is ~1 min on a GPU.

def nb_train_and_eval(cfg_path, run_name):
    """Train, then evaluate on the four development cells, all inside */notebook/."""
    log = NB_LOG_DIR/f"{run_name}.log"
    print(f"training -> {log.relative_to(REPO)}")
    r = subprocess.run([PY, "-u", "scripts/training/train_unet.py",
                        "--config", str(cfg_path), "--tag", run_name],
                       cwd=REPO, capture_output=True, text=True)
    log.write_text(r.stdout + r.stderr)
    for line in r.stdout.splitlines():
        if any(k in line for k in ("split assignment", "val  :", "train:", "WARNING", "Best epoch")):
            print("   ", line)
    if r.returncode:
        print(r.stderr[-2000:]); return None

    ck = REPO/f"checkpoints/{NB}/{run_name}/unet_best.pt"
    for cell in DEV_CELLS + [None]:
        subprocess.run([PY, "scripts/training/evaluate_unet.py",
                        "--checkpoint", str(ck), "--config", str(cfg_path),
                        "--pool", "postswap_devset",
                        "--results-log", str(NB_EVAL_LOG),        # ← sandbox log
                        *(["--filter", cell] if cell else [])],
                       cwd=REPO, capture_output=True, text=True)
    # keep the training curve in the sandbox too
    for p in (REPO/"results").glob(f"unet_training_curves_*{run_name}*.png"):
        p.rename(NB_RES_DIR/p.name)
    print(f"evaluated -> {NB_EVAL_LOG.relative_to(REPO)}")
    return ck

if RUN_TRAINING:
    ckpt = nb_train_and_eval(CFG_PATH, RUN_NAME)
else:
    print("SKIPPED (RUN_TRAINING = False).")
    print(f"Would train {RUN_NAME} and write everything under */{NB}/:")
    for d in ("configs", "checkpoints", "logs", "results"):
        print(f"   {d}/{NB}/")
    print("\nNothing touches preswap/ or postswap/.")


In [ ]:

# ── Compare sandbox runs against the real baseline ────────────────────────────
SD, FLOOR = 0.047, 2*0.047*np.sqrt(2)

def load_log(p):
    out = defaultdict(dict)
    if p.exists():
        for line in p.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                out[Path(r["checkpoint"]).parent.name][r.get("filter") or "POOLED"] = r["metrics"]["1"]
    return out

base = load_log(POST_LOG).get("postswap_ftl_cw142_s42", {})
nbruns = load_log(NB_EVAL_LOG)

if not nbruns:
    print(f"no sandbox runs yet in {NB_EVAL_LOG.relative_to(REPO)}")
elif base:
    rows = []
    for run, m in nbruns.items():
        row = {"run": run}
        for c in DEV_CELLS + ["POOLED"]:
            if c in m and c in base:
                row[c.replace("_cell","-")] = round(m[c]["dsc"], 3)
                row[f"Δ {c.replace('_cell','-')}"] = round(m[c]["dsc"] - base[c]["dsc"], 3)
        rows.append(row)
    display(pd.DataFrame(rows))
    for run, m in nbruns.items():
        if "POOLED" in m and "POOLED" in base:
            d = m["POOLED"]["dsc"] - base["POOLED"]["dsc"]
            verdict = ("INCONCLUSIVE — inside the noise floor; log as screened, not a result"
                       if abs(d) < FLOOR else "clears the noise floor — confirm with 3 seeds on BOTH arms")
            print(f"{run}: pooled Δ {d:+.3f}   {verdict}")
    print(f"\nnoise floor ±{FLOOR:.3f} (pooled sd {SD}). Sandbox runs are exploratory —")
    print("promote anything real into configs/postswap/ and re-run it properly.")


### 7d · Promote a sandbox run into the record

If a sandbox run looks real, move it up rather than repeating the work. `promote()`
does three things:

1. **Writes a proper config** into `configs/postswap/` — rebuilt from the baseline
   *text*, so the baseline's explanatory comments survive, with a header recording
   where it came from and what changed.
2. **Verifies the sandbox run actually matches that config** on every training-relevant
   field. If it doesn't, it refuses to carry the artifacts over and tells you to re-run.
   That is the guard against promoting a result produced by a config you have since
   edited.
3. **Carries the checkpoint, log, curve and eval rows across** as the seed-42 arm, so
   only the remaining seeds need running.

> **One run is not a result**, however good it looks. Measured seed-only spread is
> sd 0.047, so anything under ~0.13 pooled is inside the noise. Promotion gives you the
> seed-42 arm; `run_postswap.sh` at seeds 43 and 44 is what makes it reportable.


In [ ]:

# ── Promote a sandbox run into configs/postswap + the real record ─────────────
import shutil, re as _re
from celldform.config import load as _load

# Fields that define a run. If any differs between the sandbox config and the promoted
# config, the sandbox artifacts do NOT describe the promoted config and cannot be reused.
TRAINING_FIELDS = ("epochs","batch_size","lr","weight_decay","optimizer","scheduler",
                   "loss_type","tversky_alpha","tversky_beta","focal_gamma",
                   "class_weights","dice_exclude_background","seed","split_seed",
                   "loss_alpha","early_stopping_patience","augment_noise_std",
                   "augment_cutout_prob","augment_brightness_contrast_prob",
                   "augment_scale_prob","cudnn_deterministic","cudnn_benchmark")

def _set_yaml_value(text, section, key, value):
    """Replace `key:` inside `section:` in the baseline TEXT, keeping its comment."""
    lines = text.splitlines(keepends=True)
    in_sec, out, done = False, [], False
    for ln in lines:
        if _re.match(rf"^{section}:", ln): in_sec = True
        elif _re.match(r"^\S", ln) and not ln.startswith("#"): in_sec = False
        m = _re.match(rf"^(\s+){key}:\s*(\S.*?)(\s+#.*)?$", ln) if in_sec else None
        if m and not done:
            indent, _, comment = m.groups()
            out.append(f"{indent}{key}: {json.dumps(value)}{comment or ''}\n")
            done = True
        else:
            out.append(ln)
    return "".join(out), done

def promote(sandbox_run, new_name, why, carry_artifacts=True, WRITE=False):
    """Promote configs/notebook/<sandbox_run>.yaml -> configs/postswap/<new_name>.yaml."""
    sb_cfg_path = NB_CFG_DIR/f"{sandbox_run}.yaml"
    if not sb_cfg_path.exists():
        print(f"no sandbox config {sb_cfg_path.relative_to(REPO)}"); return
    sb = yaml.safe_load(sb_cfg_path.read_text())

    # 1 ── rebuild from the BASELINE TEXT so its comments survive
    text = BASELINE_CFG.read_text()
    changes, unhandled = {}, []
    for section in ("data", "unet", "training", "preprocessing"):
        for k, v in (sb.get(section) or {}).items():
            if (BASELINE_YAML.get(section) or {}).get(k) != v:
                text, ok = _set_yaml_value(text, section, k, v)
                (changes.__setitem__(f"{section}.{k}", v) if ok
                 else unhandled.append(f"{section}.{k}"))
    text = _re.sub(r'^checkpoint_dir:.*$', f'checkpoint_dir: "checkpoints/postswap/{new_name}"',
                   text, count=1, flags=_re.M)
    header = (f"# {new_name}\n#\n# PROMOTED from notebook run '{sandbox_run}' on "
              f"{datetime.datetime.now():%Y-%m-%d}.\n# Why: {why}\n#\n"
              "# Changed from configs/postswap/postswap_ftl_cw142_s42.yaml:\n"
              + "".join(f"#     {k}: {v!r}\n" for k, v in changes.items())
              + "#\n# STATUS: seed 42 only. Not a result until seeds 43 and 44 are run --\n"
                "# measured spread is sd 0.047, so anything under ~0.13 pooled is noise.\n\n")

    # 2 ── does the RUN match what we just wrote?
    #
    # Compare the snapshot the trainer saved INSIDE the checkpoint directory (what
    # actually trained) against the promoted config. Both are normalised through the
    # real loader first: the snapshot stores resolved values and the YAML omits
    # defaults and nests `augmentation`, so comparing the raw dicts reports
    # differences that are only shape. Comparing the promoted config against the file
    # it was BUILT FROM would always agree and catch nothing.
    out_path = REPO/"configs/postswap"/f"{new_name}.yaml"
    mismatches = []
    snap_path = REPO/f"checkpoints/{NB}/{sandbox_run}/config.yaml"
    if carry_artifacts:
        if not snap_path.exists():
            mismatches.append(f"no run snapshot at {snap_path.relative_to(REPO)} — "
                              "cannot verify what this checkpoint was trained with")
        else:
            tmp = NB_CFG_DIR/f".promote_check_{new_name}.yaml"
            tmp.write_text(header + text)
            try:
                ran, want = _load(snap_path), _load(tmp)
                for f in TRAINING_FIELDS:
                    a, b = getattr(ran.training, f, None), getattr(want.training, f, None)
                    if a != b: mismatches.append(f"training.{f}: RAN WITH {a!r}, config says {b!r}")
                for f in ("in_channels","out_channels","base_features","dropout"):
                    a, b = getattr(ran.unet, f), getattr(want.unet, f)
                    if a != b: mismatches.append(f"unet.{f}: RAN WITH {a!r}, config says {b!r}")
                if ran.data.val_split != want.data.val_split:
                    mismatches.append(f"data.val_split: RAN WITH {ran.data.val_split!r}, "
                                      f"config says {want.data.val_split!r}")
            finally:
                tmp.unlink(missing_ok=True)

    print(f"promote  {sandbox_run}  ->  configs/postswap/{new_name}.yaml")
    print(f"   changes vs baseline: {changes or '(none)'}")
    if unhandled:
        print(f"   ⚠️  could not place in the baseline text: {unhandled} — check the file by hand")
    if mismatches:
        print("\n   ⚠️  SANDBOX RUN DOES NOT MATCH THIS CONFIG — artifacts will NOT be carried over:")
        for m in mismatches: print(f"        {m}")
        print("        (the checkpoint was trained with different settings — re-run, or")
        print("         promote with carry_artifacts=False and run all three seeds fresh)")
        carry_artifacts = False

    if not WRITE:
        print("\n   DRY RUN (WRITE=False). Nothing written.")
        return out_path

    out_path.write_text(header + text)
    print(f"\n   wrote {out_path.relative_to(REPO)}")

    if carry_artifacts:
        src_ck = REPO/f"checkpoints/{NB}/{sandbox_run}"
        dst_ck = REPO/"checkpoints/postswap"/new_name
        if src_ck.exists() and not dst_ck.exists():
            shutil.copytree(src_ck, dst_ck); print(f"   checkpoint -> {dst_ck.relative_to(REPO)}")
        src_log = NB_LOG_DIR/f"{sandbox_run}.log"
        if src_log.exists():
            shutil.copy2(src_log, REPO/"logs/postswap"/f"{new_name}.log")
            print(f"   log        -> logs/postswap/{new_name}.log")
        for p in NB_RES_DIR.glob(f"*{sandbox_run}*.png"):
            shutil.copy2(p, REPO/"results/postswap"/p.name.replace(sandbox_run, new_name))
        # re-key the eval rows onto the new checkpoint path and append to the real log
        n = 0
        if NB_EVAL_LOG.exists():
            with (REPO/"results/postswap/postswap_eval_log.jsonl").open("a") as fh:
                for line in NB_EVAL_LOG.read_text().splitlines():
                    if not line.strip(): continue
                    r = json.loads(line)
                    if Path(r["checkpoint"]).parent.name != sandbox_run: continue
                    r["checkpoint"] = f"checkpoints/postswap/{new_name}/unet_best.pt"
                    r["config"] = str(out_path.relative_to(REPO))
                    r["promoted_from"] = f"notebook:{sandbox_run}"     # provenance kept
                    fh.write(json.dumps(r) + "\n"); n += 1
            print(f"   eval rows  -> {n} appended to results/postswap/ (tagged promoted_from)")

    print(f"\n   NEXT — one seed is not a result. Run the other two arms:")
    for s in (43, 44):
        print(f"        cp configs/postswap/{new_name}.yaml configs/postswap/{new_name[:-4]}_s{s}.yaml"
              if new_name.endswith("_s42") else f"        # make a seed-{s} copy of {new_name}")
    print(f"        scripts/training/run_postswap.sh <that name>")
    print(f"        python scripts/training/compare_postswap.py --seeds ...")
    return out_path

# DRY RUN by default — shows exactly what it would do and writes nothing
promote("nb_demo", "postswap_ftl_cw142_nbdemo_s42",
        why="example — replace with the real reason this run is worth keeping",
        WRITE=False)


In [ ]:
# ── Training curves (sandbox run if present, else the baseline) ─────────────────────────────────────
import re as _re
# prefer a sandbox run if one exists, else fall back to the real baseline
log = REPO/"logs/notebook"/f"{RUN_NAME}.log"
if not log.exists():
    log = REPO/"logs/postswap/postswap_ftl_cw142_s42.log"
if log.exists():
    h = defaultdict(list)
    for m in _re.finditer(r"^Epoch (\d+)/\d+\s+train_loss=([\d.]+)\s+train_DSC=([\d.]+)\s+"
                          r"val_loss=([\d.]+)\s+val_DSC=([\d.]+)", log.read_text(), _re.M):
        for k, g in zip(("train_loss","train_dsc","val_loss","val_dsc"), m.groups()[1:]):
            h[k].append(float(g))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(h["train_loss"], label="train"); ax[0].plot(h["val_loss"], label="val")
    ax[0].set_title("Segmentation loss — ALL 3 CLASSES"); ax[0].legend(); ax[0].set_xlabel("epoch")
    ax[1].plot(h["train_dsc"], label="train"); ax[1].plot(h["val_dsc"], label="val")
    ax[1].set_title("Trapped-cell DSC only"); ax[1].legend(); ax[1].set_xlabel("epoch")
    plt.tight_layout(); plt.show()
    print("The loss panel spans 3 classes; the DSC panel is class 1 alone. A gap can open")
    print("in the loss with the DSC curves overlapping — that gap is the DECOY class")
    print("(train 0.859 vs val 0.594), not the trapped cell (0.953 vs 0.934).")
else:
    print(f"no log at {log.relative_to(REPO)} — run training first")

---
## 8 · Evaluate on the development set

The only number that counts. Internal validation carries **no signal** for these
comparisons — it separated two checkpoints by 0.001 while the development set separated
the same two by 0.066.

In [ ]:
# ── Per-cell + pooled evaluation of the baseline checkpoint ───────────────────
RUN_EVAL = False     # reads the checkpoint and runs the model; ~1 min

if RUN_EVAL and BASELINE_CKPT.exists():
    for cell in DEV_CELLS + [None]:
        sh("scripts/training/evaluate_unet.py", "--checkpoint", BASELINE_CKPT,
           "--config", BASELINE_CFG, "--pool", "postswap_devset",
           *(["--filter", cell] if cell else []), show=False)
    print("appended to", POST_LOG.relative_to(REPO))

# read whatever is already logged
rows = [json.loads(l) for l in POST_LOG.read_text().splitlines() if l.strip()]
by = defaultdict(dict)
for r in rows:
    by[Path(r["checkpoint"]).parent.name][r.get("filter") or "POOLED"] = r["metrics"]["1"]

seeds = sorted(k for k in by if k.startswith("postswap_ftl_cw142_s"))
tbl = []
for c in DEV_CELLS + ["POOLED"]:
    v = [by[s][c]["dsc"] for s in seeds if c in by[s]]
    if len(v) < 2: continue
    tbl.append({"cell": c.replace("_cell","-"), **{s[-3:]: round(x,3) for s, x in zip(seeds, v)},
                "mean": round(np.mean(v),3), "sd": round(np.std(v, ddof=1),3)})
display(pd.DataFrame(tbl))

In [ ]:
# ── The noise floor — why single runs cannot settle small differences ─────────
SD = 0.047                      # pooled sd, 3 seeds, split pinned (measured 2026-09-17)
FLOOR = 2*SD*np.sqrt(2)         # ~0.133

print(f"pooled sd (3 seeds, split pinned): {SD}")
print(f"smallest believable single-run difference: {FLOOR:.3f}\n")
for name, d in [("background exclusion", .007), ("class weights [1,3,3]", .008),
                ("Focal-Tversky", .023), ("scale augmentation (SETTLED RECIPE)", .066),
                ("high-9 after the cell exchange", .245)]:
    print(f"   {d:+.3f}  {name:<38}{'below the floor — NOT a result' if d < FLOOR else 'clears the floor'}")
print("\nUse scripts/training/compare_postswap.py; it prints this verdict for you.")

In [ ]:
# ── Predictions vs ground truth, per cell ─────────────────────────────────────
from celldform.segmentation.unet import UNet

if BASELINE_CKPT.exists():
    model = UNet.from_checkpoint(BASELINE_CKPT, in_channels=1, out_channels=3,
                                 base_features=64,
                                 device="cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    dev_in = REPO/"data/preprocessed/evaluate/postswap_devset/inputs"
    dev_lb = REPO/"data/preprocessed/evaluate/postswap_devset/labels"

    fig, ax = plt.subplots(len(DEV_CELLS), 3, figsize=(9, 3*len(DEV_CELLS)))
    for i, cell in enumerate(DEV_CELLS):
        fs = sorted(dev_in.glob(f"{cell}_*.png"))
        if not fs: 
            for a in ax[i]: a.axis("off")
            continue
        fp = fs[len(fs)//2]
        img = cv2.imread(str(fp), cv2.IMREAD_GRAYSCALE)
        gt  = cv2.imread(str(dev_lb/fp.name), cv2.IMREAD_GRAYSCALE)
        pred = model.predict(img.astype(np.float32)/255.0, binary_output=True)
        inter = ((pred>0) & (gt==1)).sum()
        dsc = 2*inter/max((pred>0).sum() + (gt==1).sum(), 1)
        ax[i,0].imshow(img);        ax[i,0].set_ylabel(cell.replace("_cell","-"), fontsize=9)
        ax[i,1].imshow(gt==1);      ax[i,1].set_title("ground truth (trapped cell)", fontsize=8)
        ax[i,2].imshow(pred>0);     ax[i,2].set_title(f"prediction   DSC {dsc:.3f}", fontsize=8)
        for a in ax[i]: a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()
else:
    print("checkpoint missing — train first")

---
## 9 · Stage 4 — morphology features

The frozen U-Net runs over `03_mask_factory` with `binary_output=True`, collapsing the
3-class map to trapped-cell-only before anything downstream sees it. Multiclass is a
**training-time mechanism only**.

The feature vector is always **sorted alphabetically by name**. Changing that ordering
requires updating `RealTimePipeline.process_frame` and `scripts/inference/infer.py` in
the same commit.

In [ ]:
# ── Features from predicted masks ─────────────────────────────────────────────
from celldform.features.extractor import MorphologyExtractor
mex = MorphologyExtractor(pixel_size_um=conf.features.pixel_size_um,
                          min_area_px=conf.features.min_area_px)

if BASELINE_CKPT.exists():
    sample = sorted((FR/"03_mask_factory").glob("high_cell2_*.jpg"))[:12]
    masks, ids = [], []
    for p in sample:
        img = pre(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE))
        masks.append(model.predict(img.astype(np.float32)/(255.0 if img.max()>1 else 1.0),
                                   binary_output=True))
        ids.append(int(p.stem.split("_")[-1]))
    feats = mex.batch(masks, frame_ids=ids)
    display(feats.head())
    print("features (alphabetical):", ", ".join(sorted(c for c in feats.columns if c != "frame_id")))
    nan = feats.isna().all(axis=1).sum()
    if nan:
        print(f"\n⚠️  {nan} all-NaN rows — empty mask after the {conf.features.min_area_px}px area filter.")
        print("    Check how RealTimePipeline handles a NaN row before running end to end.")

---
## 10 · Stage 5 — HER2 classification

**Cell-level cross-validation only.** Frames from one cell share biology and laser
session; a frame-level split leaks and inflates the score. Use `groups=cell_id`.

In [ ]:
# ── Train and evaluate a classifier ───────────────────────────────────────────
from celldform.classification.classifiers import CellClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score

feat_csv = REPO/conf.data.features_csv
lab_csv  = REPO/conf.data.labels_csv
if feat_csv.exists() and lab_csv.exists():
    X = pd.read_csv(feat_csv); y = pd.read_csv(lab_csv)
    groups = X.pop("cell_id") if "cell_id" in X else None
    clf = CellClassifier(scale_features=conf.classification.scale_features,
                         random_state=conf.classification.random_state,
                         class_weight="balanced")
    if groups is not None:
        s = cross_val_score(clf, X, y.values.ravel(), groups=groups, cv=LeaveOneGroupOut())
        print(f"leave-one-cell-out accuracy: {s.mean():.3f} ± {s.std():.3f}  (n={len(s)} cells)")
    else:
        print("no cell_id column — cannot do cell-level CV; do NOT report a frame-level score")
else:
    print(f"Not built yet: {feat_csv.name} / {lab_csv.name}")
    print("\nStage 5 is the next phase of the project. To build the feature table:")
    print("  1. run the frozen U-Net over data/frames/03_mask_factory (555 frames)")
    print("  2. MorphologyExtractor.batch over the predicted masks")
    print("  3. label each row high/low from its cell's folder, keep cell_id for grouping")
    print(f"\nFactory pool: high {sum(1 for _ in (FR/'03_mask_factory').glob('high_*.jpg'))}"
          f"  low {sum(1 for _ in (FR/'03_mask_factory').glob('low_*.jpg'))}")

---
## 11 · Stage 6 — biomechanics

Deformation is **shape change over increasing laser power**, not displacement — the
cell is trapped. Frame index is the proxy for power: these videos ramp it over 50
increments.

In [ ]:
# ── Rate of deformation, and a Kelvin-Voigt fit ───────────────────────────────
from celldform.biomechanics.analysis import DeformationAnalyzer
ana = DeformationAnalyzer(metric=conf.biomechanics.metric,
                          laser_power_mW=conf.biomechanics.laser_power_mW,
                          trap_current_mA=conf.biomechanics.trap_current_mA,
                          window_s=conf.biomechanics.window_s)
try:
    if "feats" in dir() and not feats.empty and conf.biomechanics.metric in feats:
        series = feats.dropna(subset=[conf.biomechanics.metric])
        rod = ana.deformation_rate(series[conf.biomechanics.metric].values)
        print(f"metric        : {conf.biomechanics.metric}")
        print(f"rate of deform: {rod}")
        plt.figure(figsize=(7,3))
        plt.plot(series["frame_id"], series[conf.biomechanics.metric], "o-")
        plt.xlabel("frame index  (proxy for laser power)")
        plt.ylabel(conf.biomechanics.metric); plt.title("deformation vs laser power")
        plt.tight_layout(); plt.show()
    else:
        print("run the feature cell above first")
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print("Stage 6 runs after the classifier; see docs/pipeline.md.")

---
## 12 · Stage 7 — real-time inference

`RealTimePipeline` chains all six stages over a live or recorded stream. Only Stage 3
uses the GPU; everything else is CPU.

In [ ]:
# ── Assemble the full pipeline ────────────────────────────────────────────────
from celldform.integration.realtime import RealTimePipeline

if BASELINE_CKPT.exists():
    results = []
    rt = RealTimePipeline(unet=model, preprocessor=pre, extractor=mex,
                          classifier=None, analyzer=ana,
                          device="cuda" if torch.cuda.is_available() else "cpu",
                          on_result=results.append)
    demo = sorted((FR/"02_unet_holdout").glob("high_cell9_*.jpg"))[:5]
    for i, p in enumerate(demo):
        try:
            rt.process_frame(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE), frame_index=i)
        except Exception as e:
            print(f"frame {i}: {type(e).__name__}: {e}"); break
    if results:
        display(pd.DataFrame([rt.to_json_safe(r) for r in results]))
        print("\nclassifier=None here — wire a trained CellClassifier in once Stage 5 exists.")

---
## Appendix — traps this pipeline has actually fallen into

| Trap | What happens | Guard |
|---|---|---|
| **Stale preprocessing** | `preprocess_frames.py` appends without cleaning, so a cell moved between pools survives in **both** — a silent firewall break | delete the output dirs before re-running (section 6) |
| **Single-run deltas** | differences under ~0.13 DSC are noise; two findings have been retracted | `compare_postswap.py` |
| **Oversized val group** | `high_cell10` (50 frames) can take over a ~45-frame validation set on 13% of seeds, voiding the run | split log line, section 7 |
| **Comparing across the exchange** | pre- and post-17-Sept numbers describe different development sets | `preswap/` vs `postswap/` folders |
| **`change_threshold` set** | the static trapped cell gets filtered out and you extract nothing | keep it `None` |
| **Mask interpolation** | linear/cubic resize destroys the 0/1/2 label values | `INTER_NEAREST` only |
| **Renaming a run** | `postswap_eval_log.jsonl` keys on the checkpoint's parent directory name; a rename orphans its results | never rename, create a new run |

### Where to read more

`docs/status.md` (where the project stands) · `docs/workflow.md` (experiment rules) ·
`configs/README.md` and `checkpoints/README.md` (what every file is) ·
`scripts/report/README.md` (rebuilding the progress report).